In [3]:
import random
import math

# Практические задания

#### 1. Болезнь и тест. Постройте таблицу: для заболеваемости 0.01%, 1%, 10% и точности теста 90%, 99%, 99.9% — какова постериорная вероятность болезни при положительном тесте?

In [1]:
prevalences = [0.0001, 0.01, 0.1]
accuracies = [0.9, 0.99, 0.999]

results = {}

for pi in prevalences:
    results[pi] = {}

    for q in accuracies:
        sensitivity = q
        specificity = q

        false_positive_rate = 1 - specificity

        numerator = pi * sensitivity
        denominator = numerator + (1 - pi) * false_positive_rate

        if denominator == 0:
            posterior = None
        else:
            posterior = numerator / denominator

        if posterior is not None and posterior < 0:
            raise ValueError("posterior не может быть меньше 0")

        if posterior is not None and posterior > 1:
                    raise ValueError("posterior не может быть больше 1")

        results[pi][q] = posterior

# Проверка задания
pi = 0.5
q = 0.5
numerator = pi * q
denominator = numerator + (1 - pi) * (1 - q)
assert numerator / denominator == 0.5

if pi in prevalences:
     q = 1
     numerator = pi * q
     denominator = numerator + (1 - pi) * (1 - q)
     posterior = numerator / denominator
     assert posterior == 1

if pi in prevalences:
     q = 0.5
     numerator = pi * q
     denominator = numerator + (1 - pi) * (1 - q)
     posterior = numerator / denominator
     assert abs(posterior - pi) < 1e-12

# Вывод таблицы в процентах
print("Заболеваемость | " + " | ".join(f"Точность {q * 100:g}%" for q in accuracies))
print("-" * 75)

for pi in prevalences:
    values = []
    for q in accuracies:
        posterior = results[pi][q]
        values.append("не определено" if posterior is None else f"{posterior * 100:.4f}%")

    print(f"{pi * 100:g}%                  | " + " | ".join(values))


Заболеваемость | Точность 90% | Точность 99% | Точность 99.9%
---------------------------------------------------------------------------
0.01%                  | 0.0899% | 0.9804% | 9.0835%
1%                  | 8.3333% | 50.0000% | 90.9836%
10%                  | 50.0000% | 91.6667% | 99.1071%


#### 2. Парадокс Монти Холла. Симулируйте 100 000 игр. Сравните стратегии «не менять» и «менять». Какая выигрывает чаще?

In [2]:
random.seed(42)

# Количество экспериментов
N = 100000

# Инициализация счетчиков
wins_stay = 0
wins_switch = 0

doors = [1, 2, 3]

for _ in range(N):
    # Где находится приз и какую дверь выбирает игрок
    prize_door = random.randint(1, 3)
    player_choice = random.randint(1, 3)

   # Ведущий открывает дверь, которая не выбрана игроком и за которой нет приза

    candidates = [
        door for door in doors
        if door != player_choice and door != prize_door
    ]

    host_opens = random.choice(candidates)

    # Стратегия «остаться»
    final_stay = player_choice   

    # Стратегия «сменить»
    final_switch = next(
        door for door in doors
        if door != player_choice and door != host_opens
    )

    # Проверка выигрыша
    if prize_door == player_choice:
        wins_stay += 1
    else:
        wins_switch += 1

prob_stay = wins_stay / N
prob_switch = wins_switch / N

print(f"Остаться: {wins_stay} побед из {N} ({prob_stay:.2%})")
print(f"Сменить: {wins_switch} побед из {N} ({prob_switch:.2%})")

Остаться: 33252 побед из 100000 (33.25%)
Сменить: 66748 побед из 100000 (66.75%)


#### 3. Парадокс дней рождения. В группе из 23 человек какова вероятность совпадения дней рождения? Посчитайте аналитически и проверьте симуляцией.

In [4]:
random.seed(42)

# Параметры задачи
n = 23      # Людей в группе
d = 365     # Дней в году
M = 10000   # Количество симуляций

# Проверка входных данных
if n < 0:
    raise ValueError("Количество человек n не должно быть отрицательным.")
if d <= 0:
    raise ValueError("Количество дней d должно быть больше 0.")
if M <= 0:
    raise ValueError("Количество симуляций M должно быть больше 0.")

# === АНАЛИТИЧЕСКИЙ РАСЧЕТ ===
# Вычислить вероятность всех различных дней рождения
if n <= 1:
    p_match_analytic = 0.0
elif n > d:
    p_match_analytic = 1.0
else:
    p_unique = 1.0
    for k in range(n):
        p_unique *= (d - k) / d
    # Вычислить вероятность совпадения
    p_match_analytic = 1 - p_unique

# === СИМУЛЯЦИЯ МОНТЕ КАРЛО ===
# Инициализация счетчика
count_matches = 0

for _ in range(M):
    # Генерация дней рождения для группы из n человек
    birthdays = [
        random.randint(1, d)
        for _ in range(n)
    ]

    # Проверка наличия дубликатов
    if len(set(birthdays)) < n:
        count_matches += 1    

# Итоговая оценка вероятности
p_match_simulated = count_matches / M

# Оценка погрешности симуляции
standard_error = math.sqrt(p_match_simulated * (1 - p_match_simulated) / M)
margin_of_error_95 = 1.96 * standard_error

ci_low = max(0.0, p_match_simulated - margin_of_error_95)
ci_high = min(1.0, p_match_simulated + margin_of_error_95)

# Абсолютная разница между симуляцией и аналитическим результатом
absolute_error = abs(p_match_simulated - p_match_analytic)

print(f"Аналитическая вероятность: {p_match_analytic:.6f} ({p_match_analytic:.2%})")
print(f"Вероятность по симуляции: {p_match_simulated:.6f} ({p_match_simulated:.2%})")
print(f"Стандартная ошибка симуляции: {standard_error:.6f}")
print(f"Погрешность 95%:             ±{margin_of_error_95:.6f}")
print(f"Приблизительный 95% интервал: [{ci_low:.6f}, {ci_high:.6f}]")
print(f"Абсолютная разница результатов: {absolute_error:.6f}")

Аналитическая вероятность: 0.507297 (50.73%)
Вероятность по симуляции: 0.512800 (51.28%)
Стандартная ошибка симуляции: 0.004998
Погрешность 95%:             ±0.009797
Приблизительный 95% интервал: [0.503003, 0.522597]
Абсолютная разница результатов: 0.005503


#### 4. Закон больших чисел. Бросайте монетку n = 10, 100, 10000, 1000000 раз. Постройте графики, как доля орлов сходится к 0.5.

#### 5. E и Var на симуляции. Сгенерируйте 10 000 чисел из распределения Пуассона с λ=3. Проверьте, что среднее ≈ дисперсия ≈ 3.

#### 6. Ковариационная матрица. Сгенерируйте 1000 точек из 2D-нормального распределения с заданной ковариацией. Восстановите ковариационную матрицу из выборки. Должна совпасть с заданной.

#### 7. Наивный байес руками. Постройте классификатор спама на 5 простых текстах. Используйте мешок слов и формулу Байеса с предположением независимости.

#### 8. Условные вероятности на датасете. Возьмите Titanic. Посчитайте P(выжил | женщина), P(выжил | мужчина), P(выжил | 1 класс). Какая фича сильнее всего влияет?